# Week 5 Day 3 - Agno exercise

In [ ]:
import functools
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.mcp import MCPTools
from mcp import StdioServerParameters

import board

load_dotenv(override=True)

In [ ]:
MODEL = "gpt-5.4-mini"

model = OpenAIChat(id=MODEL)

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
# Agno does not expose the MCP server's stderr, so point its stdio client at DEVNULL.
import agno.tools.mcp.mcp as agno_mcp
agno_mcp.stdio_client = functools.partial(agno_mcp.stdio_client, errlog=subprocess.DEVNULL)

workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)

server = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
    cwd=str(workspace),
)

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    await worker.arun(input="Please work the pending goal on the board.")
board.show_board()

## Stream the loop live with aprint_response

In [ ]:
board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    # Same run, but Agno prints the tool-calling loop live as it happens.
    await worker.aprint_response(input="Please work the pending goal on the board.", stream=True)
board.show_board()